In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
import pickle


In [2]:
# Load sentiment scores
scores_df = pd.read_csv("./datasetFolder/t4sa_text_sentiment.tsv", sep="\t", names=["TWID", "NEG", "NEU", "POS"], skiprows=1)

# Load tweets
tweets_df = pd.read_csv("./datasetFolder/raw_tweets_text.csv")

# Merge both on ID
merged_df = pd.merge(tweets_df, scores_df, left_on="id", right_on="TWID")

# Assign label based on highest score
def get_label(row):
    scores = {"NEG": row["NEG"], "NEU": row["NEU"], "POS": row["POS"]}
    return max(scores, key=scores.get)

label_map = {"NEG": 0, "POS": 1, "NEU": 2}
merged_df["label"] = merged_df.apply(get_label, axis=1).map(label_map)

# Keep only text and label
final_df = merged_df[["text", "label"]]


In [3]:
# Balance the dataset (175,000 samples per class)
balanced_df = final_df.groupby('label', group_keys=False).apply(
    lambda x: x.sample(175000, random_state=42)
).reset_index(drop=True)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6568\4216982028.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = final_df.groupby('label', group_keys=False).apply(


In [4]:
# Define a function to clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)   # Remove URLs
    text = re.sub(r'\@\w+|\#', '', text)                   # Remove mentions and hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)                # Keep only letters and spaces
    text = re.sub(r'\s+', ' ', text).strip()               # Remove extra spaces
    return text

# Apply cleaning
balanced_df['clean_text'] = balanced_df['text'].apply(clean_text)


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Initialize TF-IDF vectorizer with max features (same as Logistic Regression)
vectorizer = TfidfVectorizer(max_features=10000)

# Fit and transform the clean text
X = vectorizer.fit_transform(balanced_df['clean_text'])

# Labels
y = balanced_df['label']

# Split into train and test (80% train, 20% test, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [6]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# Initialize Naive Bayes model
nb_model = MultinomialNB()

# Train the model
nb_model.fit(X_train, y_train)

# Predict on test data
y_pred_nb = nb_model.predict(X_test)

# Print classification report and accuracy
print("Naive Bayes Results:")
print(classification_report(y_test, y_pred_nb))
print("Accuracy:", accuracy_score(y_test, y_pred_nb))


Naive Bayes Results:
              precision    recall  f1-score   support

           0       0.85      0.93      0.89     35000
           1       0.89      0.88      0.89     35000
           2       0.91      0.83      0.86     35000

    accuracy                           0.88    105000
   macro avg       0.88      0.88      0.88    105000
weighted avg       0.88      0.88      0.88    105000

Accuracy: 0.8819333333333333
